### Library & Data import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

sns.set_style('darkgrid')

In [ ]:
!git clone "https://github.com/GeeksforgeeksDS/21-Days-21-Projects-Dataset"

### Data Info

In [ ]:
df = pd.read_csv('/content/21-Days-21-Projects-Dataset/Datasets/netflix_titles.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.isna().sum()

## General Overview-

### 1. Null values are in 5 features -  
  Director, Cast, Country, date_added, rating

### 2. type are 2 only - TV show, Movie

# **Data Cleaning**

In [ ]:
print("Can't determine the director or cast by any metrics, hence filling Unknown")

df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')

df.isnull().sum()

In [ ]:
country_mode = df['country'].mode()[0]
print("Mode of country is ",country_mode)
df['country'] = df['country'].fillna(country_mode)
df.isnull().sum()

In [ ]:
print("Missing values in date_added and rating are few, dropping it")

df.dropna(subset=['date_added', 'rating'], inplace=True)
df.isnull().sum()

# Data Transformation

In [ ]:
df.head(2)

In [ ]:
print("Transforming Date format")
df['date_added'] = pd.to_datetime(df['date_added'], format='mixed', dayfirst=False)

df.head(2)

In [ ]:
print("Adding new year and month features")
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

df.head(2)

In [ ]:
print(df.dtypes)

# EDA and Visualization

### Practice Q1- What is the distribution of content type?

In [ ]:
plt.figure(figsize=(8,6))
type_counts = df['type'].value_counts()
print(type_counts)

In [ ]:
plt.pie(type_counts, labels=type_counts.index, autopct = '%1.1f%%', startangle=140, colors=['#F5921B', '#63BDBD'])
plt.title('Proportion of Content')
plt.show()

### Outcome - More Movies are ther than TV Shows on Netflix

### Practice Q2- How has content been added over time?

In [ ]:
content_over_time_temp = df.groupby(['year_added', 'type']).size()
print(content_over_time_temp)

In [ ]:
print(content_over_time_temp.unstack().fillna(0))

In [ ]:
content_over_time = content_over_time_temp.unstack().fillna(0)

plt.figure(figsize=(14,8))
content_over_time.plot(kind='line', marker='o', figsize=(14,8))
plt.title('Content added in Netflix over the years by type')
plt.xlabel('Year added')
plt.ylabel('Number of each contnet type added')
plt.legend(title='Content type')
plt.grid(True)
plt.show()

### Outcome - 2016 to 2021 the major content was added though Netflix started from 2008.

#### Also there is a significant dipfrom 1400+ to almost 0 after 2021 - COVID-19 effect

### Practice Q3- What are the most popular genres?

In [ ]:
df.head(1)

In [ ]:
genres_df = df.assign(genre = df['listed_in'].str.split(', ')).explode('genre')
genres_df.head(10)

In [ ]:
top_genres = genres_df['genre'].value_counts().reset_index()
print(top_genres)

In [ ]:
top_genres.columns = ['genre', 'count']
print(top_genres)

In [ ]:
top_genres_plot = top_genres.head(15)

plt.figure(figsize=(12,8))
sns.barplot(y='genre', x='count', data=top_genres_plot, palette='mako', hue='genre', legend=False)
plt.title('Top 15 Genres on Netflix')
plt.xlabel('Count')
plt.ylabel('Genre')
plt.show()

### Outcome - International Movies and Dramas are the top genres

### Practice Q4- What is the distribution of content duration?

In [ ]:
movies_df = df[df['type'] == 'Movie'].copy()
tvshows_df = df[df['type'] == 'TV Show'].copy()

In [ ]:
movies_df.head(1)

In [ ]:
tvshows_df.head(1)

In [ ]:
movies_df['duration_min'] = movies_df['duration'].str.replace(' min', '').astype(int)
tvshows_df['seasons'] = tvshows_df['duration'].str.replace(' Seasons', '').str.replace(' Season', '').astype(int)

In [ ]:
movies_df.head(1)

In [ ]:
tvshows_df.head(1)

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(18,7))

sns.histplot(ax=axes[0], data=movies_df, x='duration_min', bins=50, kde=True, color='skyblue').set_title('Movie Duration Distribution (minutes)')

sns.countplot(ax=axes[1], x='seasons', data=tvshows_df, palette='rocket', order=tvshows_df['seasons'].value_counts().index, hue='seasons', legend=False).set_title('TV Show Season Distribution')

plt.show()

### Outcome - Majority of movies are in 90 to 120 mins range.
### Only limited shows had seasons continued, mostly had only 1 season

### Practice Q5- Where does the content come from? (Geographical Analysis)

In [ ]:
df.head()

In [ ]:
countries_df = df.assign(country=df['country'].str.split(', ')).explode('country')
countries_df.head()

In [ ]:
top_countries_counts = countries_df['country'].value_counts().reset_index()
top_countries_counts.columns = ['country', 'count']

In [ ]:
print(top_countries_counts)

In [ ]:
top_countries_counts_plot = top_countries_counts.head(15)

plt.figure(figsize=(12, 10))
sns.barplot(y='country', x='count', data=top_countries_counts_plot, palette='viridis', hue='country', legend=False)
plt.title('Top 15 Content Producing Countries on Netflix')
plt.xlabel('Number of Titles')
plt.ylabel('Country')
plt.show()

### Outcome- United States is largest producer of content followed by India

### Practice Q6- What are the maturity ratings of the content?

In [ ]:
df.head(5)

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(x='rating', data=df, order=df['rating'].value_counts().index, palette='crest', hue='rating', legend=False)
plt.title('Distribution of Content Ratings on Netflix')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

## Feature Engineering

In [ ]:
df['content_age'] = df['year_added'] - df['release_year']

content_age = df[df['content_age'] >= 0]

plt.figure(figsize=(14, 7))
sns.histplot(data=content_age, x='content_age', bins=50, kde=True)
plt.title('Distribution of Content Age When Added to Netflix')
plt.xlabel('Content Age (Years)')
plt.ylabel('Number of Titles')
plt.show()

### Outcome- Large spike at 0 => almost all content was added on netflix the same year it was released

## Deeper multivariate analysis

In [ ]:
top_genres = genres_df['genre'].value_counts().index[:5]
genres_movies = genres_df[(genres_df['type'] == 'Movie') & (genres_df['genre'].isin(top_genres))].copy()
genres_movies['duration_min'] = genres_movies['duration'].str.replace(' min', '').astype(int)

plt.figure(figsize=(15, 8))
sns.boxplot(data=genres_movies, x='genre', y='duration_min', palette='pastel', hue='genre', legend=False)
plt.title('Movie Duration by Top Genres')
plt.xlabel('Genre')
plt.ylabel('Duration (minutes)')
plt.xticks(rotation=45)
plt.show()

## Word Cloud from Content Description

In [ ]:
combined_text = ' '.join(df['description'])

wordcloud = WordCloud(width=800, height=400, background_color='black').generate(combined_text)
plt.figure(figsize=(15, 10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Common Words in Netflix Content Descriptions', fontsize=20)
plt.show()

# Submission Questions

### Q1- How has the distribution of content ratings changed over time?

In [ ]:
df.head(1)

In [ ]:
df['rating'].value_counts()

In [ ]:
df['rating'].isnull().sum()

In [ ]:
content_ratings_overtime = df.groupby(['year_added', 'rating']).size().unstack().fillna(0)

print(content_ratings_overtime.sum())

In [ ]:
plt.figure(figsize=(14, 8))
content_ratings_overtime.plot(kind='line', marker='o', figsize=(14, 8))
plt.title('Content Ratings Over the Years')
plt.xlabel('Year Added')
plt.ylabel('Content Rating feature')
plt.legend(title='Content Rating')
plt.grid(True)
plt.show()

### Outcome-
#### TV_MA is the rating given to most content (max 800) during 2016 to 2021
#### Second highest rating category was TV-14 (max 520)
#### Lowest rating given is NC-17 to only 3 contents

### Q2- Is there a relationship between content age and its type (Movie vs. TV Show)?

In [ ]:
df.info()

#### Type feature is object, content_age feature is int. Doing Bivariate analysis of Categorical vs Numeric

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(x="type", y="content_age", data=df)
plt.title("Content Age Distribution by Type")
plt.show()

sns.violinplot(x="type", y="content_age", data=df)
plt.title("Content Age Distribution by Type (Violin Plot)")
plt.show()

In [ ]:
sns.barplot(x="type", y="content_age", data=df, palette='viridis', hue='type', estimator=lambda x: x.mean())
plt.title("Average Content Age by Type")
plt.show()

### Outcome- TV Shows have newer content age and Movie have older content age

### Q3- Can we identify any trends in content production based on the release year vs. the year added to Netflix?

In [ ]:
df.head()

In [ ]:
plt.scatter(df["release_year"], df["content_age"], alpha=0.5)
plt.xlabel("Release Year")
plt.ylabel("Content Age")
plt.title("Relationship between Release Year and Content Age")
plt.show()

In [ ]:
release_year_feature = df['release_year']
year_added_feature = df['year_added']

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(x="release_year", y="year_added", data=df, alpha=0.5)
plt.title("Release Year vs. Year Added to Netflix")
plt.show()

In [ ]:
plt.figure(figsize=(12,7))
sns.histplot(data=df, x="release_year", y="year_added", bins=30, cbar=True)
plt.title("Content Volume by Release Year vs. Added Year")
plt.show()

### Outcome- Older content was initially added on Netflix and also added to Netflix much later after release.
### In latest years, content was quickly added to Netflix as released.

### Q4- What are the most common word pairs or phrases in content descriptions?

In [ ]:
print(df['description'])

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize, bigrams, FreqDist
import string

nltk.download('punkt')
nltk.download("punkt_tab")
nltk.download('stopwords')

In [ ]:
complete_text = " ".join(df["description"].dropna().astype(str)).lower()

tokens = word_tokenize(complete_text)
tokens = [t for t in tokens if t.isalpha()]
tokens = [t for t in tokens if t not in stopwords.words("english")]

In [ ]:
bigrams_list = list(bigrams(tokens))
print(bigrams_list)

In [ ]:
bigram_freq = FreqDist(bigrams_list)
print(bigram_freq.most_common(20))

In [ ]:
pairs_df = pd.DataFrame(bigram_freq.most_common(15), columns=["Pair", "Frequency"])
pairs_df["Pair"] = pairs_df["Pair"].apply(lambda x: " ".join(x))
plt.figure(figsize=(10,6))
sns.barplot(x="Frequency", y="Pair", data=pairs_df)
plt.title("Top 15 Most Common Word Pairs in Descriptions")
plt.show()

### Outcome- Most common word pair is 'high school'

### Q5- Who are the top directors on Netflix?

In [ ]:
df['director'].value_counts()

In [ ]:
all_directors = df['director'].str.split(",").explode().str.strip()
print(all_directors)

In [ ]:
top_directors = all_directors.value_counts().head()
print(top_directors)

### Outcome- Most are Unknow. After this, the top director is 'Jan Suter'